# Tp03 : Étude de cas Yelp

## Création du Dataset

### Import des librairie

In [160]:
from pandas import read_csv, merge, DataFrame
from numpy import nan, floor
from utils import create_data_path, parse_hours
import matplotlib.pyplot as plt
import numpy as np

Matplotlib is building the font cache; this may take a moment.


### Variable

In [161]:
export_path: str = "./restaurants_features.csv"
days: list[str] = ["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"]
data_set_path: dict[str, str] = {
    "avis"         : create_data_path("avis.csv"),
    "categories"   : create_data_path("categories.csv"),
    "checkin"      : create_data_path("checkin.csv"),
    "conseils"     : create_data_path("conseils.csv"),
    "horaires"     : create_data_path("horaires.csv"),
    "restaurants"  : create_data_path("restaurants.csv"),
    "services"     : create_data_path("services.csv"),
    "utilisateurs" : create_data_path("utilisateurs.csv")
}

c:\Users\DCS2003\Desktop\1-cegep\5-session05\5-collecte_interpretation\2-travaux_pratique\2-python\420-514-MV-TP03
c:\Users\DCS2003\Desktop\1-cegep\5-session05\5-collecte_interpretation\2-travaux_pratique\2-python\420-514-MV-TP03
c:\Users\DCS2003\Desktop\1-cegep\5-session05\5-collecte_interpretation\2-travaux_pratique\2-python\420-514-MV-TP03
c:\Users\DCS2003\Desktop\1-cegep\5-session05\5-collecte_interpretation\2-travaux_pratique\2-python\420-514-MV-TP03
c:\Users\DCS2003\Desktop\1-cegep\5-session05\5-collecte_interpretation\2-travaux_pratique\2-python\420-514-MV-TP03
c:\Users\DCS2003\Desktop\1-cegep\5-session05\5-collecte_interpretation\2-travaux_pratique\2-python\420-514-MV-TP03
c:\Users\DCS2003\Desktop\1-cegep\5-session05\5-collecte_interpretation\2-travaux_pratique\2-python\420-514-MV-TP03
c:\Users\DCS2003\Desktop\1-cegep\5-session05\5-collecte_interpretation\2-travaux_pratique\2-python\420-514-MV-TP03


### Charger Dataset

In [162]:
df_avis         = read_csv(data_set_path["avis"])
df_categories   = read_csv(data_set_path["categories"])
df_checking     = read_csv(data_set_path["checkin"])
df_conseils     = read_csv(data_set_path["conseils"])
df_horaires     = read_csv(data_set_path["horaires"])
df_restaurants  = read_csv(data_set_path["restaurants"])
df_services     = read_csv(data_set_path["services"])
df_utilisateurs = read_csv(data_set_path["utilisateurs"])

In [163]:
data = DataFrame(df_restaurants)
data = data.drop(["zone", "ferme"], axis = 1)

In [164]:
review_count = (df_avis
    .groupby("restaurant_id")
    .size()
    .reset_index(name = "review_count_total"))

In [165]:
positive_reviews = (df_avis[df_avis["etoiles"] >= 4]
    .groupby("restaurant_id")
    .size()
    .reset_index(name = "review_positive"))

In [166]:
review = merge(review_count, 
               positive_reviews, 
               left_on= "restaurant_id", 
               right_on="restaurant_id", 
               how="inner")

In [167]:
data = data.merge(review, 
             left_on  = "restaurant_id", 
             right_on = "restaurant_id", 
             how      = "inner")

In [168]:
data["review_count_total"] = (data["review_count_total"]
    .fillna(nan)
    .astype("int64"))

In [169]:
data["review_positive"] = (data["review_positive"]
    .fillna(nan)
    .astype("int64"))

In [170]:
data["positive_ratio"] = floor((data["review_positive"] / data["review_count_total"]) * 100) / 100

In [171]:
checkins_total = (df_checking.groupby("restaurant_id")
    .size()
    .reset_index(name="checkins_total"))

In [172]:
data = data.merge(checkins_total, 
             left_on = "restaurant_id", 
             right_on = "restaurant_id", 
             how = "inner")

In [173]:
data["checkins_total"] = data["checkins_total"].fillna(nan).astype("int64")

In [174]:
name_counts = data["nom"].value_counts()

In [175]:
data["is_chain"] = data["nom"].map(lambda x: name_counts[x] >= 3)

In [176]:
prix_moyen = (df_services
    .groupby("restaurant_id")["prix"]
    .mean()
    .reset_index(name="prix_moyen"))

In [177]:
data = data.merge(prix_moyen, 
                  left_on = "restaurant_id", 
                  right_on = "restaurant_id", 
                  how = "left")

In [178]:
df_elite = df_utilisateurs[df_utilisateurs["elite"].notna()]

In [179]:
df_elite = df_utilisateurs[df_utilisateurs["elite"] != "[]"]

In [180]:
elite_id = df_elite[["utilisateur_id"]]

In [181]:
elite_reviews = df_avis.merge(elite_id,
                                on="utilisateur_id",
                                how="inner")


In [182]:
elite_users_count = (elite_reviews
                     .groupby("restaurant_id")["utilisateur_id"]
                     .nunique()
                     .reset_index(name="elite_users_count"))

In [183]:
data = data.merge(elite_users_count,
    on="restaurant_id",
    how="left")

In [184]:
for day in days:
    df_horaires[day] = df_horaires[day].apply(parse_hours)

In [185]:
df_horaires["avg_open_hours"] = df_horaires[days].mean(axis=1)

In [186]:
df_horaires["avg_open_hours"] = floor(df_horaires["avg_open_hours"] * 100) / 100

In [187]:
data = data.merge(df_horaires[["restaurant_id", "avg_open_hours"]],
    on="restaurant_id",
    how="left")

In [188]:
data.to_csv(export_path)